# sgd-vanilla-from-scratch — worked example 1: One SGD step over two parameters with different gradients

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sgd-vanilla-from-scratch`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Vanilla SGD updates each parameter by subtracting learning_rate × gradient: `p -= lr * p.grad`. The update must be in-place so that external references to the parameter tensor (like state dicts) remain valid. After the update, p.grad is set to None so the next backward call starts fresh.

## Worked solution

**Step 1 — Create MiniTensor parameters.** We build two parameters with known initial values and manually set their gradients.

**Step 2 — Record values before the step.** We save the initial values so we can verify the update math: `new_val = old_val - lr * grad`.

**Step 3 — Call sgd_step.** This loops over params, applies `p.array -= lr * p.grad` in place, then sets `p.grad = None`.

**Step 4 — Verify each parameter individually.** We check both that the numeric values updated correctly and that the `.grad` attribute is now None.

**Why in-place matters.** If you wrote `p.array = p.array - lr * p.grad`, you'd rebind the attribute to a new tensor object. Any other code holding a reference to the old array would see stale values.

In [ ]:
import torch as t

t.manual_seed(42)

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.grad = None
        self.recipe = None

def sgd_step(params, lr):
    for p in params:
        if p.grad is None:
            continue
        p.array -= lr * p.grad
        p.grad = None

# Two parameters: w with grad 2.0, b with grad -0.5
w = MiniTensor(t.tensor([3.0]), requires_grad=True)
b = MiniTensor(t.tensor([1.0]), requires_grad=True)
w.grad = t.tensor([2.0])
b.grad = t.tensor([-0.5])

lr = 0.1
w_before = w.array.item()
b_before = b.array.item()

sgd_step([w, b], lr)

print(f'w: {w_before:.2f} - {lr} * 2.0 = {w_before - lr * 2.0:.2f}  got {w.array.item():.2f}')
print(f'b: {b_before:.2f} - {lr} * (-0.5) = {b_before - lr * -0.5:.2f}  got {b.array.item():.2f}')
print(f'w.grad after step: {w.grad}  (should be None)')
print(f'b.grad after step: {b.grad}  (should be None)')